# planetx — module tests

In [ ]:
%load_ext autoreload
%autoreload 2
RUN_LIVE = True

import json, tempfile, traceback, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS

warnings.filterwarnings("ignore", category=fits.verify.VerifyWarning)

RESULTS = {}
def check(name, fn):
    """Run one check; record and print ✅ / ⏳ / ❌ / 💥 without stopping the notebook."""
    try:
        out = fn()
        RESULTS[name] = "pass"; print(f"✅ {name}"); return out
    except NotImplementedError:
        RESULTS[name] = "todo"; print(f"⏳ {name}: TODO")
    except AssertionError as e:
        RESULTS[name] = "fail"; print(f"❌ {name}: {e}")
    except Exception as e:
        RESULTS[name] = "crash"; print(f"💥 {name}: {type(e).__name__}: {e}")
        print("   " + traceback.format_exc().strip().splitlines()[-3])

@contextmanager
def patched(obj, attr, value):
    """Temporarily replace obj.attr (a tiny monkeypatch)."""
    old = getattr(obj, attr)
    setattr(obj, attr, value)
    try:
        yield
    finally:
        setattr(obj, attr, old)

TMP = Path(tempfile.mkdtemp(prefix="planetx_nb_"))
CENTER = SkyCoord(133.7951, -7.2451, unit="deg")

def fake_cutout(path, center, star, size=40, scale=6.15, rot_deg=30.0, zodi=5.0, flag_pixel=None):
    """SPHEREx-like MEF (IMAGE/FLAGS/ZODI) on a ROTATED TAN WCS with a Gaussian star at `star`.
    FLAGS header uses the real SPHEREx convention: MSKN0000=TRANSIENT, MSKN0004=HOT, MSKN0021=SOURCE."""
    w = WCS(naxis=2)
    w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    w.wcs.crval = [center.ra.deg, center.dec.deg]
    w.wcs.crpix = [size / 2 + 0.5, size / 2 + 0.5]
    s, th = scale / 3600, np.radians(rot_deg)
    w.wcs.cd = [[-s * np.cos(th), s * np.sin(th)], [s * np.sin(th), s * np.cos(th)]]
    x0, y0 = w.world_to_pixel(star)
    yy, xx = np.mgrid[:size, :size]
    img = zodi + 100 * np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * 1.2 ** 2))
    flags = np.zeros((size, size), dtype=np.int32)
    if flag_pixel:
        flags[flag_pixel] = 1 << 4
    hdr = w.to_header(); hdr["BUNIT"] = "MJy/sr"
    fh = fits.Header({"MSKN0000": "TRANSIENT", "MSKM0000": 1, "MSKN0004": "HOT     ",
                      "MSKM0004": 16, "MSKN0021": "SOURCE  ", "MSKM0021": 1 << 21})
    fits.HDUList([fits.PrimaryHDU(),
                  fits.ImageHDU(img.astype(np.float32), hdr, name="IMAGE"),
                  fits.ImageHDU(flags, fh, name="FLAGS"),
                  fits.ImageHDU(np.full((size, size), zodi, np.float32), name="ZODI")]).writeto(path, overwrite=True)
    return Path(path)

from planetx import config, targets, query, cutouts, stack, export, pipeline, cli, api
from planetx.targets import Target
print("imports OK · scratch dir:", TMP)

## 1 · config.py

In [ ]:
def t_root():
    r = config.root()
    assert (r / "config" / "sources.toml").exists(), f"root() = {r} has no config/sources.toml"
    return r
check("config.root", t_root)

def t_sources():
    s = config.load_sources()
    for k in ("spherex", "grid", "cutout", "flags", "live"):
        assert k in s, f"sources.toml missing [{k}]"
    assert "spherex_dr1" in s["spherex"]["collections"]
check("config.load_sources", t_sources)

def t_targets_cfg():
    ids = [t["id"] for t in config.load_targets()]
    assert {"wise0855", "barnard", "eps-indi-b", "rw-aur", "r-dra"} <= set(ids), ids
check("config.load_targets", t_targets_cfg)

def t_dirs():
    c, w = config.cache_dir(), config.web_data_dir()
    assert c.is_dir() and c.name == "cache", c
    assert w.parts[-3:] == ("web", "public", "data"), w
check("config.cache_dir / web_data_dir", t_dirs)

## 2 · targets.py

In [ ]:
def t_pm():
    t = Target("x", "x", "", 10.0, 0.0, pmra=0.0, pmdec=10_000.0, ref_epoch=2000.0)
    assert abs(t.at(2001.0).dec.arcsec - 10.0) < 1e-6, "10 arcsec/yr north not applied"
    t2 = Target("x", "x", "", 10.0, 60.0, pmra=3600.0, ref_epoch=2000.0)   # 3.6"/yr east at dec 60
    dra = (t2.at(2001.0).ra.deg - 10.0) * 3600
    assert abs(dra - 7.2) < 1e-3, f"RA must be divided by cos(dec): got {dra:.4f} arcsec, want 7.2"
check("Target.at (proper motion)", t_pm)

def t_wrap():
    t = Target("x", "x", "", 359.9999, 0.0, pmra=3.6e6, ref_epoch=2000.0)   # +1 deg/yr
    assert 0 <= t.at(2001.0).ra.deg < 1, "RA not wrapped into [0, 360)"
check("Target.at (RA wrap)", t_wrap)

def t_info():
    d = Target("a", "A", "k", 1.0, 2.0).info()
    assert {"id", "name", "kind", "ra", "dec", "pmra", "pmdec", "ref_epoch"} <= set(d), d
    json.dumps(d, allow_nan=False)
check("Target.info", t_info)

def t_resolve_explicit():
    t = targets.resolve({"id": "w", "name": "W", "kind": "bd", "ra": 133.7951, "dec": -7.2451})
    assert (t.ra, t.dec, t.pmra, t.ref_epoch) == (133.7951, -7.2451, 0.0, 2000.0)
check("targets.resolve (explicit ra/dec)", t_resolve_explicit)

def t_adhoc():
    t = targets.adhoc(133.7951, -7.2451)
    assert t.pmra == t.pmdec == 0 and t.kind == "custom" and "133.7951" in t.id, t
check("targets.adhoc", t_adhoc)

## 3 · query.py (offline parts)

In [ ]:
check("query.detector_of", lambda: (_ for _ in ()).throw(AssertionError("wrong"))
      if query.detector_of("SPHEREx-D3") != "D3" else None)

def t_empty():
    e = query.empty()
    assert len(e) == 0 and set(query.KEEP + ["detector"]) <= set(e.colnames), e.colnames
check("query.empty", t_empty)

EP = Table({"t_min": [61169.5, 60800.7, 60990.0, 60801.1, 60991.0],     # deliberately unsorted
            "t_max": [61169.6, 60800.8, 60990.1, 60801.2, 60991.1],
            "detector": ["D1", "D1", "D2", "D2", "D1"],
            "em_min": [0.75e-6, 0.76e-6, 1.1e-6, 1.12e-6, 0.8e-6],
            "em_max": [1.0e-6, 1.09e-6, 1.6e-6, 1.65e-6, 1.05e-6]})

def t_epochs():
    out = query.assign_epochs(EP, 30)
    assert list(out["t_min"]) == sorted(EP["t_min"]), "not sorted by t_min"
    assert list(out["epoch"]) == [0, 0, 1, 1, 2], list(out["epoch"])
    assert "epoch" not in EP.colnames, "modified the input table — return a copy"
    return out
EPO = check("query.assign_epochs", t_epochs)

def t_summary():
    s = query.epoch_summary(EPO)
    assert [x["id"] for x in s] == ["A", "B", "C"], s
    assert s[0]["label"] == "May 2025", s[0]["label"]
    assert s[0]["n_images"] == 2 and s[0]["mjd_start"] == 60800.7 and s[0]["mjd_end"] == 60801.2
    assert 2025.3 < s[0]["year_mid"] < 2025.4, s[0]["year_mid"]
check("query.epoch_summary", t_summary)

def t_ranges():
    r = query.detector_ranges(EP)
    assert r == {"D1": [0.75, 1.09], "D2": [1.1, 1.65]}, r
check("query.detector_ranges", t_ranges)

## 4 · cutouts.py (offline parts)

In [ ]:
def t_url():
    u_ = cutouts.cutout_url("https://x/img.fits", CENTER, 0.05)
    assert u_ == "https://x/img.fits?center=133.795100,-7.245100&size=0.05000", u_
check("cutouts.cutout_url", t_url)

def t_cache_key():
    a = cutouts.cutout_url("https://x/img.fits", CENTER, 0.05)
    b = cutouts.cutout_url("https://x/img.fits", SkyCoord(10, 10, unit="deg"), 0.05)
    pa, pb = cutouts.cache_path(a, TMP), cutouts.cache_path(b, TMP)
    assert pa != pb, "same exposure at two positions collides in the cache"
    assert pa == cutouts.cache_path(a, TMP) and pa.suffix == ".fits" and pa.parent == TMP
check("cutouts.cache_path", t_cache_key)

def t_fetch_cached():
    url = "https://example.invalid/never-downloaded.fits"
    p = cutouts.cache_path(url, TMP)
    fake_cutout(p, CENTER, CENTER)
    assert cutouts.fetch_cutout(url, TMP) == p, "should return the cached file without downloading"
check("cutouts.fetch_cutout (cache hit)", t_fetch_cached)

def t_fetch_many_failure():
    good = "https://example.invalid/never-downloaded.fits"
    bad = "https://example.invalid/does-not-exist.fits"
    out = cutouts.fetch_many([good, bad], TMP, workers=2)
    assert out[0] == cutouts.cache_path(good, TMP) and out[1] is None, out
check("cutouts.fetch_many (order + failure -> None)", t_fetch_many_failure)

## 5 · stack.py

In [ ]:
def t_grid():
    g = stack.make_grid(CENTER, 25, 6.15)
    assert g.wcs.cdelt[0] < 0 < g.wcs.cdelt[1], "need east-left (CDELT1<0), north-up (CDELT2>0)"
    x, y = g.world_to_pixel(CENTER)
    assert abs(x - 12) < 1e-6 and abs(y - 12) < 1e-6, f"target at ({x:.3f},{y:.3f}), want (12,12)"
    assert tuple(g.pixel_shape) == (25, 25)
check("stack.make_grid", t_grid)

def t_flag_bits():
    h = fits.Header({"MSKN0004": "HOT     ", "MSKM0004": 16, "MSKD0004": "Hot pixel",
                     "MSKN0021": "SOURCE  ", "MSKM0021": 1 << 21, "OTHER": 1})
    assert stack.flag_bits(h) == {"HOT": 4, "SOURCE": 21}, stack.flag_bits(h)
check("stack.flag_bits", t_flag_bits)

def t_bad_mask():
    h = fits.Header({"MSKN0004": "HOT", "MSKN0021": "SOURCE"})
    f = np.array([[0, 1 << 4], [1 << 21, (1 << 4) | (1 << 21)]])
    m = stack.bad_mask(f, h, ["HOT"])
    assert m.tolist() == [[False, True], [False, True]], m.tolist()
    assert not stack.bad_mask(f, fits.Header(), ["HOT"]).any(), "no MSKN keys -> mask nothing"
check("stack.bad_mask", t_bad_mask)

def t_load():
    p = fake_cutout(TMP / "load.fits", CENTER, CENTER, zodi=5.0, flag_pixel=(5, 5))
    data, wcs, meta = stack.load_cutout(p, bad_flags=["HOT"])
    assert np.isnan(data[5, 5]) and meta["masked"] == 1, meta
    assert abs(np.nanmedian(data)) < 1e-3, "ZODI not subtracted"
    assert wcs.naxis == 2 and meta["bunit"] == "MJy/sr"
    d2, _, _ = stack.load_cutout(p, bad_flags=None)
    assert np.isfinite(d2).all(), "bad_flags=None must not mask"
check("stack.load_cutout", t_load)

def t_align():
    """THE alignment test: rotated input, star 3 px east + 2 px south -> right grid pixel."""
    star = CENTER.spherical_offsets_by(3 * 6.15 * u.arcsec, -2 * 6.15 * u.arcsec)
    p = fake_cutout(TMP / "align.fits", CENTER, star)
    g = stack.make_grid(CENTER, 25, 6.15)
    img = stack.to_grid(p, g, 25)
    assert img.shape == (25, 25), img.shape
    r, c = np.unravel_index(np.nanargmax(img), img.shape)
    ex, ey = g.world_to_pixel(star)
    assert abs(c - ex) <= 1 and abs(r - ey) <= 1, f"peak at col {c}, row {r}; expected ~({ex:.1f}, {ey:.1f})"
    assert c < 12, "star is EAST, must be LEFT of centre"
    assert abs(np.nanmedian(img)) < 1e-6, "background not subtracted"
check("stack.to_grid (alignment)", t_align)

def t_offgrid():
    far = SkyCoord(134.5, -7.2451, unit="deg")          # cutout nowhere near the grid
    p = fake_cutout(TMP / "far.fits", far, far)
    img = stack.to_grid(p, stack.make_grid(CENTER, 25, 6.15), 25)
    assert np.isnan(img).all(), "off-footprint pixels must be NaN, not 0"
check("stack.to_grid (off footprint -> NaN)", t_offgrid)

def t_stack():
    a, b, c = np.ones((3, 3)), np.full((3, 3), 3.0), np.full((3, 3), np.nan)
    img, n = stack.stack_epoch([a, b, c, None])
    assert n == 2 and np.allclose(img, 2.0), (n, img)
    assert stack.stack_epoch([None, c]) == (None, 0)
check("stack.stack_epoch", t_stack)

## 6 · export.py

In [ ]:
def t_jsonable():
    out = export.to_jsonable(np.array([[1.0, 2.0], [3.0, np.nan]]))
    assert out == [[3.0, None], [1.0, 2.0]], f"row flip / NaN wrong: {out}"
    assert export.to_jsonable(np.array([[1.234567, np.inf]])) == [[1.235, None]]
    assert type(out[0][0]) is float, "must be Python float, not numpy"
check("export.to_jsonable", t_jsonable)

def t_write():
    p = TMP / "sub" / "t.json"
    export.write_json(p, {"x": [[1.0, None]]})
    assert json.loads(p.read_text()) == {"x": [[1.0, None]]}
    try:
        export.write_json(p, {"bad": float("nan")})
    except ValueError:
        pass
    else:
        raise AssertionError("NaN was written — use allow_nan=False")
    assert json.loads(p.read_text()) == {"x": [[1.0, None]]}, "failed write clobbered the old file"
check("export.write_json", t_write)

## 7 · pipeline.py (network mocked)

In [ ]:
SOURCES_T = {"spherex": {"collections": ["x"], "epoch_gap_days": 30, "obs_epoch": 2025.9,
                         "pixel_scale_arcsec": 6.15},
             "grid": {"size_px": 25}, "cutout": {"margin": 1.5, "workers": 2},
             "flags": {"bad": ["HOT"]}, "live": {"max_per_group": 4}}
PM = 2 * 6.15 * 1000 / 0.5            # +pmra = EAST, 2 px per half year
MOVER = Target("t", "Test", "test", CENTER.ra.deg, CENTER.dec.deg, pmra=PM, ref_epoch=2025.9)
ROWS = Table({
    "obs_id": ["a", "b", "c", "d", "e"], "access_url": ["u1", "u2", "u3", "u4", "u5"],
    "energy_bandpassname": ["SPHEREx-D1", "SPHEREx-D4", "SPHEREx-D1", "SPHEREx-D4", "SPHEREx-D1"],
    "t_min": [60800.0, 60800.0, 61169.0, 61169.0, 61169.2],
    "t_max": [60800.1, 60800.1, 61169.1, 61169.1, 61169.3],
    "em_min": [0.75e-6, 2.4e-6, 0.75e-6, 2.4e-6, 0.75e-6],
    "em_max": [1.1e-6, 3.8e-6, 1.1e-6, 3.8e-6, 1.1e-6],
    "s_pixel_scale": [6.15] * 5, "dataproduct_subtype": ["science"] * 5,
    "obs_collection": ["spherex_dr1"] * 5, "detector": ["D1", "D4", "D1", "D4", "D1"]})

def fake_fetch(urls, cache, workers=8, progress=None):
    out = []
    for u_ in urls:
        key = u_.split("?")[0]
        mjd = 60800.0 if key in ("u1", "u2") else 61169.0
        year = 2025.0 + (mjd - 60676.0) / 365.25
        p = TMP / f"pipe_{key}.fits"
        if not p.exists():
            fake_cutout(p, CENTER, MOVER.at(year))
        out.append(p)
    return out

def run_pipeline(**kw):
    with patched(query, "find_images", lambda *a, **k: ROWS.copy()), \
         patched(cutouts, "fetch_many", fake_fetch):
        return pipeline.build_target(MOVER, SOURCES_T, TMP, **kw)

def t_pipe_shape():
    d = run_pipeline()
    json.dumps(d, allow_nan=False)
    for k in ("target", "grid", "units", "generated_at", "collections", "detectors",
              "epochs", "n_images", "n_used"):
        assert k in d, f"missing top-level key {k!r}"
    assert d["n_images"] == 5 and d["n_used"] == 5, (d["n_images"], d["n_used"])
    assert [e["id"] for e in d["epochs"]] == ["A", "B"]
    e = d["epochs"][1]
    assert set(e["frames"]) == {"D1", "D4", "ALL"}, set(e["frames"])
    assert e["frames"]["D1"]["n_images"] == 2 and e["frames"]["ALL"]["n_images"] == 3
    assert len(e["frames"]["ALL"]["pixels"]) == 25 and len(e["frames"]["ALL"]["pixels"][0]) == 25
    assert set(d["detectors"]) == {"D1", "D4"}
    return d
PIPE = check("pipeline.build_target (shape + JSON-safe)", t_pipe_shape)

def t_pipe_track():
    d = PIPE or run_pipeline()
    for e in d["epochs"]:
        img = np.array(e["frames"]["ALL"]["pixels"], dtype=float)
        r, c = np.unravel_index(np.nanargmax(img), img.shape)          # screen coords
        p = e["predicted_px"]
        assert abs(c - p["x"]) <= 1 and abs(r - p["y"]) <= 1, f"{e['id']}: peak ({c},{r}) vs predicted ({p['x']},{p['y']})"
    a, b = d["epochs"][0]["predicted_px"], d["epochs"][1]["predicted_px"]
    assert b["x"] < a["x"], "moving EAST must move LEFT on screen"
check("pipeline.build_target (star on predicted_px)", t_pipe_track)

def t_pipe_cap():
    d = run_pipeline(max_per_group=1)
    assert d["epochs"][1]["frames"]["D1"]["n_images"] == 1, "max_per_group not applied"
check("pipeline.build_target (max_per_group)", t_pipe_cap)

def t_pipe_empty():
    with patched(query, "find_images", lambda *a, **k: query.empty()):
        d = pipeline.build_target(MOVER, SOURCES_T, TMP)
    assert d["epochs"] == [] and d["n_images"] == 0 and d["detectors"] == {}, d
    json.dumps(d, allow_nan=False)
check("pipeline.build_target (no images)", t_pipe_empty)

## 8 · cli.py (pipeline mocked, writes to a temp dir)

In [ ]:
def t_cli():
    out = TMP / "webdata"
    fake = lambda t, *a, **k: {"epochs": [{"id": "A"}], "n_images": 7, "n_used": 7}
    fake_resolve = lambda cfg: Target(cfg["id"], cfg["name"], cfg.get("kind", ""), 1.0, 2.0)
    with patched(config, "web_data_dir", lambda: out), \
         patched(cli, "build_target", fake), patched(cli.targets, "resolve", fake_resolve):
        cli.build(["wise0855"])
        cli.build(["barnard"])                                # must MERGE, not overwrite
    idx = json.loads((out / "targets.json").read_text())
    assert {e["id"] for e in idx} == {"wise0855", "barnard"}, idx
    assert json.loads((out / "targets" / "wise0855.json").read_text())["n_images"] == 7
    man = json.loads((out / "manifest.json").read_text())
    assert {"generated_at", "collections", "counts"} <= set(man), man
check("cli.build", t_cli)

## 9 · api.py (pipeline mocked)

In [ ]:
def t_api():
    from fastapi.testclient import TestClient
    calls = []
    def fake(t, *a, **k):
        calls.append((t.id, k.get("max_per_group")))
        return {"target": t.info(), "epochs": [], "n_images": 0}
    with patched(config, "cache_dir", lambda: TMP / "apicache"), patched(api, "build_target", fake):
        (TMP / "apicache").mkdir(exist_ok=True)
        c = TestClient(api.app)
        assert c.get("/health").json() == {"ok": True}
        r = c.get("/target", params={"ra": 133.7951, "dec": -7.2451})
        assert r.status_code == 200 and r.json()["target"]["kind"] == "custom", r.text
        c.get("/target", params={"ra": 133.7951, "dec": -7.2451})          # 2nd call -> cache
        assert len(calls) == 1, f"cache not used: built {len(calls)} times"
        assert calls[0][1] == 4, f"live cap not applied: {calls[0]}"
        assert c.get("/target", params={"ra": 10, "dec": 100}).status_code == 422
        h = c.get("/health", headers={"Origin": "https://example.com"}).headers
        assert h.get("access-control-allow-origin") == "*", "CORS middleware missing"
check("api /health, /target, cache, CORS", t_api)

## 10 · LIVE — IRSA + SIMBAD (set `RUN_LIVE = False` to skip)

In [ ]:
WISE = SkyCoord(133.7951, -7.2451, unit="deg")
LIVE_IMAGES = None

def t_live_find():
    global LIVE_IMAGES
    t = query.assign_epochs(query.find_images(WISE, config.load_sources()["spherex"]["collections"]), 30)
    assert len(t) >= 258, f"{len(t)} rows (dr1 alone gave 258 this morning)"
    assert len(set(t["epoch"])) >= 3 and set(t["detector"]) == {f"D{i}" for i in range(1, 7)}
    LIVE_IMAGES = t
    print(f"   {len(t)} images, epochs {[s['label'] for s in query.epoch_summary(t)]}")

def t_live_cutout():
    url = cutouts.cutout_url(LIVE_IMAGES["access_url"][0], WISE, 0.05)
    p = cutouts.fetch_cutout(url, config.cache_dir())
    with fits.open(p) as h:
        names = [x.name for x in h]
        print("   extensions:", names)
        print("   BUNIT:", h["IMAGE"].header.get("BUNIT"))
        if "FLAGS" in names:
            print("   flag planes:", stack.flag_bits(h["FLAGS"].header) or "NONE (no MSKN keys!)")
        shape = h["IMAGE"].data.shape
    assert "IMAGE" in names, f"extension names are {names}"
    assert 20 <= shape[0] <= 40, f"0.05 deg should be ~29 px, got {shape} — size units not degrees?"

def t_live_resolve():
    for cfg in config.load_targets():
        t = targets.resolve(cfg)
        c = t.at(2025.9)
        print(f"   {cfg['id']:11s} ra={c.ra.deg:9.4f} dec={c.dec.deg:+8.4f}  pm=({t.pmra:7.0f},{t.pmdec:7.0f}) mas/yr")

if RUN_LIVE:
    check("LIVE find_images (WISE 0855)", t_live_find)
    if LIVE_IMAGES is not None:
        check("LIVE cutout (units + extensions)", t_live_cutout)
    check("LIVE resolve all curated targets", t_live_resolve)
else:
    print("skipped")

## 11 · LIVE end-to-end + the visual check
Background stars must sit still across epochs; the target should sit on the red cross (its predicted position).

In [ ]:
def t_live_build():
    cfg = next(c for c in config.load_targets() if c["id"] == "wise0855")
    d = pipeline.build_target(targets.resolve(cfg), config.load_sources(), config.cache_dir(),
                              max_per_group=3, progress=lambda i, n: print(f"\r   cutouts {i}/{n}", end=""))
    print()
    json.dumps(d, allow_nan=False)
    assert len(d["epochs"]) >= 3 and all("ALL" in e["frames"] for e in d["epochs"])
    return d

LIVE = check("LIVE build_target (WISE 0855, capped)", t_live_build) if RUN_LIVE else None

if LIVE:
    import matplotlib.pyplot as plt
    eps = LIVE["epochs"]
    fig, axes = plt.subplots(1, len(eps), figsize=(4 * len(eps), 4), squeeze=False)
    for ax, e in zip(axes[0], eps):
        img = np.array(e["frames"]["ALL"]["pixels"], dtype=float)
        ax.imshow(img, origin="upper", cmap="gray", interpolation="nearest",
                  vmin=np.nanpercentile(img, 5), vmax=np.nanpercentile(img, 99.5))
        ax.plot(e["predicted_px"]["x"], e["predicted_px"]["y"], "r+", ms=14, mew=1.5)
        ax.set_title(f"{e['id']} · {e['label']} · n={e['frames']['ALL']['n_images']}")
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(LIVE["target"]["name"]); plt.show()

## Summary

In [ ]:
from collections import Counter
c = Counter(RESULTS.values())
print(f"✅ {c['pass']}   ⏳ {c['todo']}   ❌ {c['fail']}   💥 {c['crash']}   (of {len(RESULTS)})")
for name, r in RESULTS.items():
    if r != "pass":
        print(f"  {dict(todo='⏳', fail='❌', crash='💥')[r]} {name}")
if c["pass"] == len(RESULTS):
    print("\nAll green — package done. Last check is yours: does the plot look right?")